In [1]:
import scanpy as sc
import torch
from pathlib import Path

from lightning.fabric.accelerators import find_usable_cuda_devices

from gctsvg.preprocessing import common, st
from gctsvg.model.ae import GCNDGIAE
from gctsvg.pipeline.denoiser import STDataModule, STDenoiser

from tqdm.auto import tqdm

/opt/conda/envs/ctSVG/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/opt/conda/envs/ctSVG/lib/python3.12/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [2]:
decomp_dir = Path('spatialscope_output/Human Kidney Cancer Interface')

In [3]:
adata_decomp = sc.read(decomp_dir / 'sp_adata.h5ad')
adata_decomp

AnnData object with n_obs × n_vars = 2047 × 2215
    obs: 'in_tissue', 'array_row', 'array_col', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mt_frac', 'donor_id', 'region', 'n_counts', 'cell_count'
    var: 'feature_types', 'genome', 'SYMBOL', 'mt', 'n_cells_by_counts-6800STDY12499411', 'mean_counts-6800STDY12499411', 'log1p_mean_counts-6800STDY12499411', 'pct_dropout_by_counts-6800STDY12499411', 'total_counts-6800STDY12499411', 'log1p_total_counts-6800STDY12499411', 'estimated_be'
    uns: 'cell_locations', 'sample_colors', 'spatial'
    obsm: 'image_features', 'spatial'

In [4]:
common.preprocess(adata_decomp)

In [5]:
accelerator = "gpu"
accelerator

'gpu'

In [6]:
strategy = "ddp_notebook"
strategy

'ddp_notebook'

In [7]:
# dataset_decomp = st.create_graph_data(adata_decomp, device=device)
# dataloader_decomp = DataLoader([dataset_decomp])
dm = STDataModule(adata_decomp)
dm.prepare_data()

In [8]:
n_feats_st = adata_decomp.n_vars
n_code_st = 200

num_epochs=1000
batch_size=adata_decomp.n_obs

In [9]:
devices = 2
devices

2

In [10]:
ae = GCNDGIAE(n_feats_st, n_code_st)
denoiser = STDenoiser(ae, 
                      num_epochs = num_epochs, 
                      accelerator = accelerator,
                      devices = devices,
                      strategy = strategy
                     )


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


GAT(2215, 200, num_layers=3)
GAT(200, 2215, num_layers=3)
DeepGraphInfomax(200)
GCNDGIAE(
  (encoder): GAT(2215, 200, num_layers=3)
  (decoder): GAT(200, 2215, num_layers=3)
  (DGI): DeepGraphInfomax(200)
)
gpu
2
ddp_notebook
[0, 1]


In [11]:
# Default used by PyTorch is "highest", but to properly utilize nvidia h100 nvl cores, the property is set to high
# torch.set_float32_matmul_precision("high")

denoiser.train(dm)

ProcessRaisedException: 

-- Process 0 terminated with the following error:
Traceback (most recent call last):
  File "/opt/conda/envs/ctSVG/lib/python3.12/site-packages/torch/multiprocessing/spawn.py", line 90, in _wrap
    fn(i, *args)
  File "/opt/conda/envs/ctSVG/lib/python3.12/site-packages/lightning/pytorch/strategies/launchers/multiprocessing.py", line 173, in _wrapping_function
    results = function(*args, **kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/envs/ctSVG/lib/python3.12/site-packages/lightning/pytorch/trainer/trainer.py", line 599, in _fit_impl
    self._run(model, ckpt_path=ckpt_path)
  File "/opt/conda/envs/ctSVG/lib/python3.12/site-packages/lightning/pytorch/trainer/trainer.py", line 968, in _run
    self.strategy.setup_environment()
  File "/opt/conda/envs/ctSVG/lib/python3.12/site-packages/lightning/pytorch/strategies/ddp.py", line 153, in setup_environment
    super().setup_environment()
  File "/opt/conda/envs/ctSVG/lib/python3.12/site-pack

In [ ]:
output = denoiser.denoise(dm, return_predictions=True)

In [ ]:
output

In [ ]:
output[0][0].shape

In [ ]:
output[0][1].min()

In [ ]:
adata_decomp.X.max()

In [ ]:
output[0][1].max()